In [1]:
import torch
from sklearn import linear_model
import os
import numpy as np
from utils import ProcessFoldData, MSE, BIOT, scale, SVD
from scipy.stats import wilcoxon

import warnings 
warnings.filterwarnings("ignore")

# DEFAULT FILE PATHS
# datasets = "../datasets/"
output = "../output/"
datasets = "../datasets/layers10_big"
try: os.mkdir(output)
except: pass

print("Default file paths:-------------")
print(f"datasets: {datasets}")
print(f"output: {output}")
print("--------------------------------")



# DEFAULT PARAMETERS
nLambdas = 10
minLambda = 0.0001
maxLambda = 3.5
K = 4            # no of folds used for cross validation
sigThresh = .05   # sigma threshold
maxiter = 200

print("Default parameters:-------------")
print(f"nLambdas: {nLambdas}")
print(f"minLambda: {minLambda}")
print(f"maxLambda: {maxLambda}")
print(f"Number of folds: {K}")
print(f"sigThresh: {sigThresh}")
print(f"maxiter: {maxiter}")
print("--------------------------------")



# PYTORCH ENVIRONMENT VARIABLES
# device = torch.device("cuda")
device = torch.device("cpu")

print(f"Device: {device}")



# DATASET PARAMETERS FOR RANDOM DATA
Bsz = 100
Edim = 384
Fdim = 21

# Loading from a file
file = True
if file:
  Embeddings = torch.tensor(np.genfromtxt(f"{datasets}/embeddings.csv", delimiter=',', dtype='float64'), device=device)
  Features =  torch.tensor(np.genfromtxt(f"{datasets}/features.csv", delimiter=',', skip_header=1, dtype='float64'), device=device)

else:
  Embeddings = torch.rand(Bsz,Edim).to(torch.float64).to(device)
  Features = torch.rand(Bsz,Fdim).to(torch.float64).to(device)


assert Embeddings.shape[1] == Edim
# assert Features.shape[1] == Fdim





##############################################
#### Run BIOT for different lambda values ####
##############################################

print("Selection of lambda in progress...")

# Define lambda vector, feature vector, and embedding vector
lambdaVals = torch.exp(torch.linspace(np.log(minLambda), np.log(maxLambda), nLambdas)).to(device) / np.sqrt(Features.shape[1])
print(f"Lambda values: {lambdaVals}")

# Split data into K folds such that each foldid has the indexes to use
foldIds = torch.split(torch.randperm(Features.size(0)), Features.size(0) // K)

Default file paths:-------------
datasets: ../datasets/layers10_big
output: ../output/
--------------------------------
Default parameters:-------------
nLambdas: 10
minLambda: 0.0001
maxLambda: 3.5
Number of folds: 4
sigThresh: 0.05
maxiter: 200
--------------------------------
Device: cpu
Selection of lambda in progress...
Lambda values: tensor([2.2942e-05, 7.3370e-05, 2.3465e-04, 7.5043e-04, 2.4000e-03, 7.6755e-03,
        2.4547e-02, 7.8505e-02, 2.5107e-01, 8.0295e-01])


In [2]:
clf = linear_model.Lasso(alpha=0.1, fit_intercept=False)
# intializing the rotation matrix
clf.coef_ = torch.zeros(Edim, 19, dtype=torch.float64, device=device)

In [3]:
# preprocess embeddings and features
Features_norm, Embeddings_norm, Features_test, Embeddings_test = ProcessFoldData(X = Embeddings, Fe = Features, testId = foldIds[1], CV=True)

print(F"Training data size {Features_norm.size()} and Testing data size {Features_test.size()}")


Training data size torch.Size([5, 19]) and Testing data size torch.Size([5117, 19])


In [4]:
from utils import SVD

In [5]:

for iter in range(1000):
    W = torch.tensor( clf.coef_.T, device=Embeddings.device)
    Rotation = SVD(Features_norm, W, Embeddings_norm)
    
    Y = torch.matmul(Embeddings_norm,Rotation)
    clf.fit(Features_norm.cpu(),Y.cpu())
    
    
    Yp= clf.predict(Features_test.cpu())
    Y = torch.matmul(Embeddings_test,Rotation)
    print(clf.score(Features_test.cpu(), Y.cpu()))

    mse = torch.mean((Y.cpu() - Yp)**2) 
    reg = np.sum(np.abs(clf.coef_))
    mse_error = mse + reg
    print(f"Iteration {iter} : MSE error {mse_error} MSE {mse} Reg {reg}")
    print(W.sum(), Rotation.sum())

0.004097543579804298
Iteration 0 : MSE error 3.6993718802666145 MSE 0.007864889804267118 Reg 3.691506990462347
tensor(0., dtype=torch.float64) tensor(384., dtype=torch.float64)


NameError: name 'clf' is not defined